In [ ]:
!pip install thefuzz python-Levenshtein

In [ ]:
# Running outside Colab: files are expected under /src
pass


In [ ]:
#All relevant imports for the notebook
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import requests
import datetime
from datetime import datetime
import sqlite3
import numpy as np
from scipy import stats
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from thefuzz import process
import geopandas as gpd
from shapely.geometry import Point


In [ ]:
import pandas as pd
from pathlib import Path

# Use project-relative paths (works when cwd is project root; /src/ fails on Windows)
SRC = Path("src")
def _path(name):
    p = SRC / name
    if not p.exists() and "Arrests" in name and " (1)" not in name:
        alt = SRC / "NYPD_Arrests_Data__Historic_ (1).csv"
        return str(alt) if alt.exists() else str(p)
    return str(p)

airbnb_nyc_df = pd.read_csv(_path("AB_NYC_2019.csv"))
nypd_shooting_df = pd.read_csv(_path("NYPD_Shooting_Incident_Data__Historic_.csv"))
ny_tree_census_df = pd.read_csv(_path("new_york_tree_census_2015.csv"))
total_population_df = pd.read_csv(_path("Total Population.csv"), header=4)
arrest_data_df = pd.read_csv(_path("NYPD_Arrests_Data__Historic_.csv"))
nta_pop_df = pd.read_csv(_path("New_York_City_Population_By_Neighborhood_Tabulation_Areas.csv"))
nyc_gdf = gpd.read_file(_path("nycgeo.json"))
"""
display(ny_tree_census_df.head())
display(nypd_shooting_df.head())
display(airbnb_nyc_df.head())
display(Total_Population_df.head())
"""

#**Most impactful factors affecting Airbnb pricing in New York City**

**Hypothesis: Greenery, Reviews, and Crime Rate affect Airbnb prices**

As Airbnb grows in popularity across the world, getting an accurate estimate of the price of Airbnbs will help new Airbnb owners to better price their houses. We will explore the relationship between various factors of the Airbnb and its price. We believe that crime rate, greenery around the borough and the reviews of the Airbnb will have the greatest impact on its price.

#**Data source identification and exploration(Prepare)**

In order to estimate the price of Airbnbs, we have extracted various data points such as room type, number of reviews, availability, boroughs, along with the three factors in our hypothesis. We have selected three datasets which are connected by the 5 types of boroughs in New York City.

List of datasets:
- [New York City Airbnb Open Data (2019) ](https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data)
- [NYPD Arrest Data (2006-2019) ](https://www.kaggle.com/datasets/thaddeussegura/nypd-arrest-data-20062019)
- [New York City Shooting Incident Dataset](https://www.kaggle.com/datasets/thaddeussegura/new-york-city-shooting-dataset)
- [Tree Census in New York City](https://www.kaggle.com/datasets/nycparks/tree-census)
- [Total Population in New York City](https://data.cccnewyork.org/data/map/97/total-population#97/a/3/147/132/a/a)

In [ ]:
def map_coordinates_to_neighbourhood(df):
    import geopandas as gpd
    import pandas as pd

    temp_df = df.copy()

    # Detect latitude / longitude columns case-insensitively.
    lower_to_original = {c.lower(): c for c in temp_df.columns}
    lat_col = lower_to_original.get('latitude')
    lon_col = lower_to_original.get('longitude')

    if lat_col is None or lon_col is None:
        raise ValueError("Input dataframe must contain latitude/longitude columns.")

    # Coerce to numeric and keep only valid coordinate rows for spatial join.
    temp_df[lat_col] = pd.to_numeric(temp_df[lat_col], errors='coerce')
    temp_df[lon_col] = pd.to_numeric(temp_df[lon_col], errors='coerce')
    valid_mask = (
        temp_df[lat_col].between(-90, 90)
        & temp_df[lon_col].between(-180, 180)
        & temp_df[lat_col].notna()
        & temp_df[lon_col].notna()
    )

    valid_df = temp_df.loc[valid_mask].copy()

    # Initialize neighbourhood to preserve original row count/order.
    temp_df['neighbourhood'] = pd.NA
    if valid_df.empty:
        return temp_df

    gdf = gpd.GeoDataFrame(
        valid_df,
        geometry=gpd.points_from_xy(valid_df[lon_col], valid_df[lat_col]),
        crs='EPSG:4326',
    )

    nyc_gdf_local = nyc_gdf[['geometry', 'NTAName']].copy()
    if nyc_gdf_local.crs is None:
        nyc_gdf_local = nyc_gdf_local.set_crs('EPSG:4326')
    elif nyc_gdf_local.crs.to_string() != 'EPSG:4326':
        nyc_gdf_local = nyc_gdf_local.to_crs('EPSG:4326')

    joined = gpd.sjoin(gdf, nyc_gdf_local, how='left', predicate='within')

    mapped = fuzzy_map_neighbourhood(joined, 'NTAName')
    temp_df.loc[mapped.index, 'neighbourhood'] = mapped['neighbourhood']

    return temp_df


def fuzzy_map_neighbourhood(df, target_column):
    from thefuzz import process

    temp_df = df.copy()
    airbnb_unique_hoods = [
        n for n in airbnb_nyc_df['neighbourhood'].unique()
        if n != 'Co-op City'
    ]

    source_hoods = temp_df[target_column].dropna().unique().tolist()

    fuzz_map = {}
    for hood in source_hoods:
        match, score = process.extractOne(hood, airbnb_unique_hoods)
        fuzz_map[hood] = match if score >= 75 else None

    temp_df['neighbourhood'] = temp_df[target_column].map(fuzz_map)
    return temp_df

###DS1: New York City Airbnb Open Data(2019)

Description: The NYC Airbnb Open Data dataset, downloaded on 28/01/2026, contains detailed listing activity and metrics including geographical locations, prices, and room types across New York City. Specifically, it provides neighbourhood_group (the five boroughs) and neighbourhood data, which offer an ideal level of granularity for analyzing market density and local pricing trends. The dataset also includes the last_review date, allowing for temporal analysis of listing popularity and guest activity. The following code imports the data into a DataFrame and displays the first few rows to illustrate the structure of the rental market data.

In [ ]:
#We first take a look at the firsst 5 rows of the dataset
airbnb_nyc_df.head()
airbnb_nyc_df.info()

In [ ]:
description_df = airbnb_nyc_df[['price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']].describe().round(1)
description_df = description_df.transpose()
description_df = description_df.rename(columns={
    'count': 'Number of Listings',
    'mean': 'Average Value',
    'std': 'Standard Deviation',
    'min': 'Minimum Value',
    '25%': '25th Percentile',
    '50%': 'Median Value',
    '75%': '75th Percentile',
    'max': 'Maximum Value'
})
display(description_df)

In [ ]:
# Check for total number of null values for each column
#
empty_records = airbnb_nyc_df.isnull().sum()
empty_records[empty_records > 0]

In [ ]:
airbnb_nyc_df['neighbourhood'] = airbnb_nyc_df['neighbourhood'].apply(lambda x: x.split(',')[0].split('-')[0].strip())
# Exclude 'Co-op City' from the neighbourhood column
filtered_airbnb_df = airbnb_nyc_df[airbnb_nyc_df['neighbourhood'] != 'Co-op City']
average_price_by_neighbourhood = filtered_airbnb_df.groupby('neighbourhood')['price'].mean().reset_index()
display(average_price_by_neighbourhood.head(20))

In [ ]:
airbnb_nyc_df['neighbourhood'].nunique()

###DS2: New York City Shooting Incident(2019)

Description: The Kaggle dataset, focusing on historic shooting incidents, provides a detailed log of public safety events including the exact date, time, and geographic coordinates of each occurrence in 2019. More precisely, borough-level data, which offers the granularity that we require. This allows for the creation of "safety proximity" features that can significantly influence localized pricing models. The following code imports the data into a DataFrame and displays the first few rows, offering an initial look at the incident reports and their spatial attributes.

In [ ]:
nypd_shooting_df.head()

In [ ]:
nypd_shooting_df.info()

In [ ]:
total_population_df.head()

In [ ]:
nypd_shooting_df = map_coordinates_to_neighbourhood(nypd_shooting_df)

In [ ]:
# Calculate value counts for each matched neighborhood
incident_counts = nypd_shooting_df['neighbourhood'].value_counts().reset_index()
incident_counts.columns = ['neighbourhood', 'incident_count']

# Display the top 20 neighborhoods by incident count
display(incident_counts.head(20))

In [ ]:
# 1. Prepare unique Airbnb neighborhoods (excluding Co-op City)
airbnb_unique_hoods = [n for n in airbnb_nyc_df['neighbourhood'].unique() if n != 'Co-op City']

# 2. Extract NTA population reference (Year 2010)
nta_2010 = nta_pop_df[nta_pop_df['Year'] == 2010].groupby('NTA Name')['Population'].sum().reset_index()
nta_names = nta_2010['NTA Name'].tolist()

# 3. Perform Fuzzy Matching (75% threshold)
mapping_results = []
for hood in airbnb_unique_hoods:
    match, score = process.extractOne(hood, nta_names)
    pop_val = nta_2010.loc[nta_2010['NTA Name'] == match, 'Population'].values[0] if score >= 75 else None
    mapping_results.append({'Airbnb Neighbourhood': hood, 'Population (2010 Census)': pop_val})

# 4. Create and display the final mapping table
airbnb_pop_map_df = pd.DataFrame(mapping_results).sort_values('Airbnb Neighbourhood')

print(f'Created population mapping for {len(airbnb_pop_map_df)} neighborhoods.')
display(airbnb_pop_map_df.head(20))

In [ ]:
# 1. Merge incident counts with the population mapping
# airbnb_pop_map_df contains the population per Airbnb neighbourhood
# incident_counts contains total shootings per airbnb_matched_neighbourhood

neighbourhood_stats = incident_counts.merge(
    airbnb_pop_map_df,
    left_on='neighbourhood',
    right_on='Airbnb Neighbourhood',
    how='inner'
)

# 2. Calculate shootings per 10,000 people
# We ensure we don't divide by zero and only include valid population data
neighbourhood_stats = neighbourhood_stats[neighbourhood_stats['Population (2010 Census)'] > 0]

neighbourhood_stats['shootings_per_10000'] = (
    neighbourhood_stats['incident_count'] / neighbourhood_stats['Population (2010 Census)']
) * 10000

# 3. Sort by the new metric
neighbourhood_stats = neighbourhood_stats.sort_values(by='shootings_per_10000', ascending=False)

# Display the top results
print("Neighborhoods with the highest shooting density (per 10,000 residents):")
display(neighbourhood_stats[['neighbourhood', 'incident_count', 'Population (2010 Census)', 'shootings_per_10000']].head(20))

###DS3 and DS4: NYPD Arrest Data and Total Population

Description for NYPD Arrest Data: The Kaggle dataset, encompassing historic NYPD arrest records from 2006 to 2024, provides a comprehensive record of every arrest effected in New York City. This dataset includes highly specific geospatial coordinates and other markers such as date and time, which are vital for a long-term analysis of neighborhood safety evolution. More precisely, it contains the borough and precinct, which are at the exact level of granularity required. For our Airbnb smart pricing project, this allows us to track how the safety profile of a specific borough and its market value—has shifted over nearly two decades. The following code imports the data into a DataFrame and displays the first few rows, highlighting the offense categories and location descriptors used in our predictive model.

Description for Total Population Data: The provided dataset, sourced from the U.S. Census Bureau, contains historic Total Population figures for New York City across various years, including our target year of 2019. The data is organized by Location, providing the borough-level granularity necessary to analyze market density and demand. This allows for a more nuanced understanding of Airbnb pricing by enabling us to calculate "per capita" metrics for crime or amenity availability. The following code imports the data into a DataFrame and displays the first few rows, showing the population counts and the specific community districts or neighborhoods covered.

In [ ]:
arrest_data_df.head()

In [ ]:
arrest_data_df.describe()

In [ ]:
arrest_data_df.info()

In [ ]:
arrest_data_df = map_coordinates_to_neighbourhood(arrest_data_df)

print(arrest_data_df)

In [ ]:
total_population_df[5:].info()

In [ ]:
total_population_df = pd.read_csv(str(Path("src") / "Total Population.csv"), header=4)
total_population_df.columns = total_population_df.iloc[0]
total_population_df = total_population_df[1:].reset_index(drop=True)
total_population_df = total_population_df.rename(columns={
    'Location': 'Borough',
    'TimeFrame': 'Year',
    'Data': 'Population'
})
total_population_df['Year'] = pd.to_numeric(total_population_df['Year'])
total_population_df['Population'] = pd.to_numeric(total_population_df['Population'])
target_boroughs = ['Manhattan', 'Queens', 'Brooklyn', 'Bronx', 'Staten Island']
target_years = range(2015, 2020)
filtered_population = total_population_df[
    (total_population_df['Borough'].isin(target_boroughs)) &
    (total_population_df['Year'].isin(target_years))
]
display(filtered_population[['Borough', 'Year', 'Population']])

In [ ]:
# Arrest density per 10,000 residents, by year
# NYPD arrests dataset uses these coordinate columns: 'Latitude' and 'Longitude'.
arrest_cols_needed = ['ARREST_DATE', 'Latitude', 'Longitude']
arrest_input = arrest_data_df[arrest_cols_needed].copy()

print("Mapping arrests to neighbourhood using Latitude/Longitude...")
arrest_mapped = map_coordinates_to_neighbourhood(arrest_input)
arrest_mapped = arrest_mapped.dropna(subset=['neighbourhood']).copy()

# 1. Parse ARREST_DATE to extract year
arrest_mapped['year'] = pd.to_datetime(arrest_mapped['ARREST_DATE'], format='%m/%d/%Y', errors='coerce').dt.year
arrest_mapped = arrest_mapped.dropna(subset=['year'])
arrest_mapped['year'] = arrest_mapped['year'].astype(int)

# 2. Count arrests per neighbourhood per year
arrest_counts_by_year = arrest_mapped.groupby(['neighbourhood', 'year']).size().reset_index(name='arrest_count')

# 3. Merge with population (2010 Census) for density calculation
arrest_density_by_year = arrest_counts_by_year.merge(
    airbnb_pop_map_df,
    left_on='neighbourhood',
    right_on='Airbnb Neighbourhood',
    how='inner'
)
arrest_density_by_year = arrest_density_by_year[arrest_density_by_year['Population (2010 Census)'] > 0]
arrest_density_by_year['arrests_per_10000'] = (
    arrest_density_by_year['arrest_count'] / arrest_density_by_year['Population (2010 Census)']
) * 10000

# 4. Display summary: top neighbourhoods by arrests_per_10000 (averaged across years)
arrest_summary = arrest_density_by_year.groupby('neighbourhood').agg(
    total_arrests=('arrest_count', 'sum'),
    mean_arrests_per_10000=('arrests_per_10000', 'mean'),
    years_covered=('year', 'nunique')
).reset_index().sort_values('mean_arrests_per_10000', ascending=False)

print("Arrest density by neighbourhood (mean arrests per 10,000 residents across years):")
display(arrest_summary.head(20))
print("\nArrest density by neighbourhood and year:")
display(arrest_density_by_year[['neighbourhood', 'year', 'arrest_count', 'Population (2010 Census)', 'arrests_per_10000']].head(25))

#**DS 5 Tree Census**

Description: The 2015 NYC Street Tree Census dataset, a landmark "citizen science" project known as TreesCount!, provides a comprehensive digital inventory of 666,134 street trees mapped across all five boroughs. This dataset is organized by individual tree records and can be organised by borough level. More precisely, it contains qualitative assessments of tree health and stewardship, which serve as proxies for neighborhood maintenance and resident engagement at the exact level of granularity required to categorize listings by street-level appeal.For our smart pricing project, this allows us to track how the density of trees in a specific boroughcan affect property prices. The following code imports the data into a DataFrame and displays the first few rows, showing the diameter at breast height (tree_dbh) and environmental indicators used in our predictive model.

In [ ]:
# 1. Apply fuzzy mapping to align tree census 'nta_name' with Airbnb neighborhoods
# Note: 'nta_name' is the target column in ny_tree_census_df
tree_census_mapped = fuzzy_map_neighbourhood(ny_tree_census_df, 'nta_name')

# 2. Exclude 'Co-op City' and records that didn't meet the 75% threshold
tree_census_mapped = tree_census_mapped[tree_census_mapped['neighbourhood'].notnull()]

# 3. Calculate total number of trees per matched neighborhood
tree_counts_per_neighborhood = tree_census_mapped.groupby('neighbourhood').size().reset_index(name='total_trees')

# 4. Sort and display the results
tree_counts_per_neighborhood = tree_counts_per_neighborhood.sort_values(by='total_trees', ascending=False)
display(tree_counts_per_neighborhood.head(20))

In [ ]:
import geopandas as gpd

# 1. Load and project GeoJSON for accurate area calculation
if 'nyc_gdf' not in globals():
    nyc_gdf = gpd.read_file(str(Path("src") / "nycgeo.json"))

nyc_projected = nyc_gdf.to_crs(epsg=2263)
nyc_projected['area_sq_miles'] = nyc_projected['geometry'].area / (5280**2)

# 2. Extract area data
raw_area_df = nyc_projected[['NTAName', 'area_sq_miles']].copy()

# 3. Apply fuzzy mapping to align with Airbnb neighborhoods
# We use the previously defined fuzzy_map_neighbourhood function
mapped_area_df = fuzzy_map_neighbourhood(raw_area_df, 'NTAName')

# 4. Clean up: Remove records that didn't match and aggregate area
# (In case multiple NTAs map to the same Airbnb neighborhood)
neighborhood_area_df = mapped_area_df.dropna(subset=['neighbourhood'])
neighborhood_area_df = neighborhood_area_df.groupby('neighbourhood')['area_sq_miles'].sum().reset_index()

# 5. Display the results
print("Square Area mapped to Airbnb Neighborhoods:")
display(neighborhood_area_df.sort_values(by='area_sq_miles', ascending=False).head(20))

In [ ]:
# Tree density per neighbourhood (trees per square mile)
tree_density_df = tree_counts_per_neighborhood.merge(
    neighborhood_area_df,
    on='neighbourhood',
    how='inner'
)

# Avoid divide-by-zero
tree_density_df = tree_density_df[tree_density_df['area_sq_miles'] > 0]

tree_density_df['trees_per_sq_mile'] = (
    tree_density_df['total_trees'] / tree_density_df['area_sq_miles']
)

tree_density_df = tree_density_df.sort_values('trees_per_sq_mile', ascending=False)

print('Tree density by neighbourhood (trees per sq mile):')
display(tree_density_df[['neighbourhood', 'total_trees', 'area_sq_miles', 'trees_per_sq_mile']].head(20))

In [ ]:
# 1. Get the list of unique Airbnb neighborhoods
airbnb_unique_hoods = set([n for n in airbnb_nyc_df['neighbourhood'].unique() if n != 'Co-op City'])

# 2. Get the list of successfully mapped neighborhoods
mapped_hoods = set(neighborhood_area_df['neighbourhood'].unique())

# 3. Find the difference
unmapped_hoods = airbnb_unique_hoods - mapped_hoods

# 4. Display the results
print(f"Total unique Airbnb neighborhoods: {len(airbnb_unique_hoods)}")
print(f"Neighborhoods mapped to area data: {len(mapped_hoods)}")
print(f"Neighborhoods NOT included: {len(unmapped_hoods)}")

if len(unmapped_hoods) > 0:
    print("\nSample of neighborhoods not included:")
    print(list(unmapped_hoods)[:20])

In [ ]:
ny_tree_census_df.head()

In [ ]:
ny_tree_census_df['boroname'].value_counts()

In [ ]:
# 1. Prepare unique Airbnb neighborhoods (excluding Co-op City)
airbnb_unique_hoods = [n for n in airbnb_nyc_df['neighbourhood'].unique() if n != 'Co-op City']

# 2. Extract NTA population reference (Year 2010)
nta_2010 = nta_pop_df[nta_pop_df['Year'] == 2010].groupby('NTA Name')['Population'].sum().reset_index()
nta_names = nta_2010['NTA Name'].tolist()

# 3. Perform Fuzzy Matching (75% threshold)
mapping_results = []
for hood in airbnb_unique_hoods:
    match, score = process.extractOne(hood, nta_names)
    pop_val = nta_2010.loc[nta_2010['NTA Name'] == match, 'Population'].values[0] if score >= 75 else None
    mapping_results.append({'Airbnb Neighbourhood': hood, 'Population (2010 Census)': pop_val})

# 4. Create and display the final mapping table
airbnb_pop_map_df = pd.DataFrame(mapping_results).sort_values('Airbnb Neighbourhood')

print(f'Created population mapping for {len(airbnb_pop_map_df)} neighborhoods.')
display(airbnb_pop_map_df.head(20))